# Markov Transition Matrix + Monte Carlo Model

Interactive test notebook for `src/models/markov_model.py`.

Sections:
1. Setup & load real BTC data
2. Discretization — visualize price-to-state mapping
3. Transition matrix — build, validate, heatmap
4. Monte Carlo simulation — forward paths
5. P(YES) estimation — sweep across strikes
6. MarkovModel.predict() — full live-interface test
7. Sensitivity analysis — n_states, smoothing, n_paths
8. Model vs market comparison
9. Slot-by-slot backtest on historical data

## 1. Setup

In [ ]:
import os, sys
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.models.markov_model import (
    discretize,
    build_transition_matrix,
    validate_transition_matrix,
    simulate_paths,
    estimate_p_yes,
    compare_model_vs_market,
    MarkovModel,
)

%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

print("imports OK")

In [ ]:
# Load the most recent BTC 1-second data.
# Auto-detect latest dated folder with a btc_live CSV.
import glob

btc_files = sorted(glob.glob(os.path.join(REPO_ROOT, "data/*/btc_live_1s*.csv")))
BTC_CSV = btc_files[-1] if btc_files else None
print("Using:", BTC_CSV)

btc_df = pd.read_csv(BTC_CSV)
btc_df = btc_df.sort_values("timestamp").reset_index(drop=True)
print(f"{len(btc_df)} rows, columns: {list(btc_df.columns)}")
btc_df.head(3)

In [ ]:
# Also load the orderbook snapshots for the market-comparison section.
ob_files = sorted(glob.glob(os.path.join(REPO_ROOT, "data/*/live_orderbook_snapshots*.csv")))
OB_CSV = ob_files[-1] if ob_files else None
print("Using:", OB_CSV)

ob_df = pd.read_csv(OB_CSV)
ob_df = ob_df.sort_values(["slot_ts", "elapsed_s"]).reset_index(drop=True)
print(f"{len(ob_df)} rows, {ob_df['slot_ts'].nunique()} slots")
ob_df.head(3)

## 2. Discretization

In [ ]:
# Use a 10-minute window of real BTC close prices.
WINDOW = 600  # seconds
prices = btc_df["close"].values[-WINDOW:].astype(float)
print(f"Window: {len(prices)} ticks, range ${prices.min():.2f} – ${prices.max():.2f}")

N_STATES = 50
states, edges = discretize(prices, n_states=N_STATES)
midpoints = 0.5 * (edges[:-1] + edges[1:])

print(f"States: {len(np.unique(states))} occupied out of {N_STATES}")
print(f"Bin width: ${(edges[1] - edges[0]):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: price series colored by state
ax = axes[0]
ax.scatter(range(len(prices)), prices, c=states, cmap="viridis", s=2, alpha=0.7)
ax.set_xlabel("tick")
ax.set_ylabel("BTC price ($)")
ax.set_title("Price series colored by Markov state")

# Right: state histogram
ax = axes[1]
ax.hist(states, bins=range(N_STATES + 1), edgecolor="black", linewidth=0.3)
ax.set_xlabel("State index")
ax.set_ylabel("Count")
ax.set_title(f"State occupancy ({len(np.unique(states))} / {N_STATES} occupied)")

plt.tight_layout()
plt.show()

## 3. Transition Matrix

In [ ]:
SMOOTHING = 1e-6
T = build_transition_matrix(states, N_STATES, smoothing=SMOOTHING)

# Validate
warnings = validate_transition_matrix(T)
if warnings:
    print("WARNINGS:", warnings)
else:
    print("Transition matrix valid (all checks passed)")

print(f"Shape: {T.shape}")
print(f"Row sums: min={T.sum(axis=1).min():.10f}, max={T.sum(axis=1).max():.10f}")
print(f"Sparsity (cells < 1e-4): {(T < 1e-4).sum()} / {T.size} ({100*(T < 1e-4).sum()/T.size:.1f}%)")
print(f"Max transition prob: {T.max():.4f}")

In [ ]:
# Heatmap of the transition matrix (log-scale for visibility).
# Zoom into the occupied region for clarity.
occupied = np.unique(states)
lo_s, hi_s = occupied.min(), occupied.max()
pad = max(2, (hi_s - lo_s) // 10)
sl = slice(max(0, lo_s - pad), min(N_STATES, hi_s + pad + 1))

fig, ax = plt.subplots(figsize=(8, 7))
T_sub = T[sl, sl]
im = ax.imshow(
    T_sub,
    cmap="hot_r",
    norm=mcolors.LogNorm(vmin=max(T_sub[T_sub > 0].min(), 1e-8), vmax=T_sub.max()),
    aspect="auto",
    origin="lower",
)
ax.set_xlabel("To state")
ax.set_ylabel("From state")
ax.set_title(f"Transition matrix (log scale, states {sl.start}–{sl.stop-1})")
plt.colorbar(im, ax=ax, label="P(transition)")
plt.tight_layout()
plt.show()

In [ ]:
# Dominant transition direction per state (mean next state - current state).
drift = np.array([np.average(range(N_STATES), weights=T[i]) - i for i in range(N_STATES)])

fig, ax = plt.subplots(figsize=(10, 3))
colors = ["green" if d > 0 else "red" for d in drift[sl]]
ax.bar(range(sl.start, sl.stop), drift[sl], color=colors, width=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("State")
ax.set_ylabel("Expected drift (states)")
ax.set_title("Per-state expected drift (green = upward, red = downward)")
plt.tight_layout()
plt.show()

## 4. Monte Carlo Simulation

In [ ]:
START_STATE = states[-1]
N_STEPS = 60       # ~60 seconds forward
N_PATHS = 5000

rng = np.random.default_rng(42)
paths = simulate_paths(T, start_state=START_STATE, n_steps=N_STEPS, n_paths=N_PATHS, rng=rng)
print(f"Simulated {N_PATHS} paths x {N_STEPS + 1} steps from state {START_STATE}")
print(f"Terminal states: mean={paths[:, -1].mean():.1f}, std={paths[:, -1].std():.1f}")

In [ ]:
# Convert state paths to price paths for plotting.
price_paths = midpoints[np.clip(paths, 0, len(midpoints) - 1)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sample paths
ax = axes[0]
n_show = min(200, N_PATHS)
for i in range(n_show):
    ax.plot(price_paths[i], alpha=0.05, color="steelblue", linewidth=0.5)
# Overlay mean path
mean_path = price_paths.mean(axis=0)
ax.plot(mean_path, color="red", linewidth=2, label="mean path")
ax.axhline(prices[-1], color="black", linestyle="--", linewidth=1, label=f"current ${prices[-1]:.2f}")
ax.set_xlabel("Step")
ax.set_ylabel("Price ($)")
ax.set_title(f"{n_show} simulated paths ({N_STEPS} steps)")
ax.legend()

# Right: terminal price distribution
ax = axes[1]
terminal_prices = price_paths[:, -1]
ax.hist(terminal_prices, bins=40, edgecolor="black", linewidth=0.3, alpha=0.7)
ax.axvline(prices[-1], color="red", linestyle="--", linewidth=2, label=f"strike = current ${prices[-1]:.2f}")
ax.set_xlabel("Terminal price ($)")
ax.set_ylabel("Count")
ax.set_title("Terminal price distribution")
ax.legend()

plt.tight_layout()
plt.show()

## 5. P(YES) Estimation — Strike Sweep

In [ ]:
# Sweep strike price across the observed range and compute P(YES) for each.
strike_range = np.linspace(prices.min() - 5, prices.max() + 5, 100)
p_yes_curve = [estimate_p_yes(paths, edges, s) for s in strike_range]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(strike_range, p_yes_curve, linewidth=2, color="steelblue")
ax.axvline(prices[-1], color="red", linestyle="--", label=f"current ${prices[-1]:.2f}")
ax.axhline(0.5, color="gray", linestyle=":", linewidth=0.8)
ax.set_xlabel("Strike price ($)")
ax.set_ylabel("P(YES) = P(terminal >= strike)")
ax.set_title("P(YES) vs Strike — Markov MC")
ax.legend()
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

# P(YES) at current price
p_at_current = estimate_p_yes(paths, edges, prices[-1])
print(f"P(YES) at strike = current price: {p_at_current:.4f}")

## 6. MarkovModel.predict() — Full Interface Test

In [ ]:
# Build a realistic snapshot from the loaded data.
ts_arr = btc_df["timestamp"].values[-600:].astype(float)
px_arr = btc_df["close"].values[-600:].astype(float)
now_ts = float(ts_arr[-1])
strike = float(px_arr[-1])  # at-the-money

snapshot = {
    "btc_prices": list(zip(ts_arr.tolist(), px_arr.tolist())),
    "strike_price": strike,
    "slot_expiry_ts": now_ts + 60,  # 60s to expiry
    "now_ts": now_ts,
}

model = MarkovModel(n_states=50, n_paths=5000, smoothing=1e-6, seed=42)
result = model.predict(snapshot)

print(f"Status:        {result.feature_status}")
print(f"P(YES):        {result.prob_yes:.4f}")
print(f"P(NO):         {result.prob_no:.4f}")
print(f"Model version: {result.model_version}")
print()
print("Last features:")
for k, v in model.last_features.items():
    print(f"  {k:25s} = {v}")

In [ ]:
# Test error paths
model2 = MarkovModel()

for label, snap in [
    ("insufficient history", {"btc_prices": [(1, 100)] * 5}),
    ("missing strike", {"btc_prices": snapshot["btc_prices"], "slot_expiry_ts": now_ts + 60}),
    ("missing expiry", {"btc_prices": snapshot["btc_prices"], "strike_price": 100.0}),
]:
    r = model2.predict(snap)
    print(f"{label:25s} -> status={r.feature_status}, prob_yes={r.prob_yes}")

## 7. Sensitivity Analysis

In [ ]:
# Sweep n_states and smoothing, measure P(YES) stability.
results = []
for ns in [10, 20, 30, 50, 80, 100]:
    for sm in [0.0, 1e-8, 1e-6, 1e-4, 1e-2]:
        m = MarkovModel(n_states=ns, n_paths=3000, smoothing=sm, seed=42)
        r = m.predict(snapshot)
        results.append({"n_states": ns, "smoothing": sm,
                        "p_yes": r.prob_yes, "status": r.feature_status})

sens_df = pd.DataFrame(results)
sens_df

In [ ]:
# Pivot table: P(YES) by n_states and smoothing.
pivot = sens_df.pivot_table(index="n_states", columns="smoothing", values="p_yes")

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto", vmin=0.3, vmax=0.7)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{c:.0e}" for c in pivot.columns], rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("Smoothing")
ax.set_ylabel("n_states")
ax.set_title("P(YES) sensitivity to n_states and smoothing")
# Annotate cells
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.3f}", ha="center", va="center", fontsize=8)
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# n_paths convergence: how many paths until P(YES) stabilizes?
path_counts = [100, 500, 1000, 2000, 5000, 10000, 20000]
p_by_npaths = []
for np_ in path_counts:
    m = MarkovModel(n_states=50, n_paths=np_, smoothing=1e-6, seed=42)
    r = m.predict(snapshot)
    p_by_npaths.append(r.prob_yes)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(path_counts, p_by_npaths, "o-", color="steelblue")
ax.set_xscale("log")
ax.set_xlabel("n_paths")
ax.set_ylabel("P(YES)")
ax.set_title("MC convergence: P(YES) vs n_paths")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("P(YES) range:", f"{min(p_by_npaths):.4f} – {max(p_by_npaths):.4f}")

## 8. Model vs Market Comparison

In [ ]:
# Grab the latest UP-side orderbook mid as the market price.
latest_slot = ob_df["slot_ts"].max()
latest_up = ob_df[(ob_df["slot_ts"] == latest_slot) & (ob_df["side"] == "up")]
if not latest_up.empty:
    market_yes = float(latest_up.iloc[-1]["mid"])
else:
    market_yes = 0.50
    print("No UP orderbook data — using default 0.50")

print(f"Market YES mid: {market_yes:.4f}")
print(f"Markov P(YES): {result.prob_yes:.4f}")
print()

comparison = compare_model_vs_market(result.prob_yes, market_yes)
for k, v in comparison.items():
    print(f"  {k:25s} = {v}")

In [ ]:
# Visualize: model probability vs market price across simulated market prices.
market_prices = np.linspace(0.10, 0.90, 50)
edge_yes_arr = [result.prob_yes - mp for mp in market_prices]
edge_no_arr = [(1 - result.prob_yes) - (1 - mp) for mp in market_prices]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(market_prices, edge_yes_arr, label="Edge YES", color="green", linewidth=2)
ax.plot(market_prices, edge_no_arr, label="Edge NO", color="red", linewidth=2)
ax.axhline(0.02, color="gray", linestyle=":", linewidth=0.8, label="min edge (2%)")
ax.axhline(-0.02, color="gray", linestyle=":", linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(market_yes, color="orange", linestyle="--", label=f"actual market = {market_yes:.2f}")
ax.fill_between(market_prices, 0.02, max(edge_yes_arr), alpha=0.1, color="green")
ax.set_xlabel("Market YES price")
ax.set_ylabel("Edge")
ax.set_title(f"Edge vs market price (model P(YES) = {result.prob_yes:.4f})")
ax.legend()
ax.set_ylim(-0.5, 0.5)
plt.tight_layout()
plt.show()

## 9. Slot-by-Slot Backtest

Run the Markov model on each historical 5-minute slot and compare its P(YES) against the market mid and the actual outcome.

In [ ]:
# Build a slot-level backtest.
SLOT_INTERVAL = 300
ts_all = btc_df["timestamp"].values.astype(float)
px_all = btc_df["close"].values.astype(float)

# Get unique slots from the orderbook data.
slot_list = sorted(ob_df["slot_ts"].unique())
print(f"{len(slot_list)} slots available")

backtest_rows = []
model_bt = MarkovModel(n_states=50, n_paths=3000, smoothing=1e-6, seed=42)

for slot_ts in slot_list:
    expiry = slot_ts + SLOT_INTERVAL
    # Decision point: 120s into the slot (slot_ts + 120).
    decision_ts = slot_ts + 120
    tte = expiry - decision_ts

    # BTC prices up to decision time.
    mask = (ts_all >= decision_ts - 600) & (ts_all <= decision_ts)
    ts_window = ts_all[mask]
    px_window = px_all[mask]
    if len(ts_window) < 30:
        continue

    strike = float(px_all[np.searchsorted(ts_all, float(slot_ts), side="right") - 1])
    if strike <= 0:
        continue

    snap = {
        "btc_prices": list(zip(ts_window.tolist(), px_window.tolist())),
        "strike_price": strike,
        "slot_expiry_ts": expiry,
        "now_ts": decision_ts,
    }
    r = model_bt.predict(snap)
    if r.prob_yes is None:
        continue

    # Actual outcome: price at expiry vs strike.
    exp_idx = np.searchsorted(ts_all, float(expiry), side="right") - 1
    if exp_idx < 0 or exp_idx >= len(px_all):
        continue
    actual_price = float(px_all[exp_idx])
    outcome = 1 if actual_price >= strike else 0

    # Market mid at decision time from the orderbook.
    slot_up = ob_df[(ob_df["slot_ts"] == slot_ts) & (ob_df["side"] == "up")]
    close_to_120 = slot_up[(slot_up["elapsed_s"] >= 110) & (slot_up["elapsed_s"] <= 130)]
    market_mid = float(close_to_120.iloc[-1]["mid"]) if not close_to_120.empty else 0.50

    backtest_rows.append({
        "slot_ts": slot_ts,
        "strike": strike,
        "actual_price": actual_price,
        "outcome": outcome,
        "p_markov": r.prob_yes,
        "market_mid": market_mid,
    })

bt_df = pd.DataFrame(backtest_rows)
print(f"Backtested {len(bt_df)} slots")
bt_df.head(10)

In [ ]:
if len(bt_df) > 0:
    from sklearn.metrics import brier_score_loss, log_loss

    y = bt_df["outcome"].values
    p_mk = np.clip(bt_df["p_markov"].values, 1e-6, 1 - 1e-6)
    p_mkt = np.clip(bt_df["market_mid"].values, 1e-6, 1 - 1e-6)

    print(f"{'Metric':<15} {'Markov':>10} {'Market':>10}")
    print(f"{'Brier':<15} {brier_score_loss(y, p_mk):>10.4f} {brier_score_loss(y, p_mkt):>10.4f}")
    print(f"{'LogLoss':<15} {log_loss(y, p_mk):>10.4f} {log_loss(y, p_mkt):>10.4f}")
    print(f"{'Accuracy':<15} {((p_mk > 0.5).astype(int) == y).mean():>10.4f} {((p_mkt > 0.5).astype(int) == y).mean():>10.4f}")
    print(f"{'N slots':<15} {len(bt_df):>10}")
else:
    print("No slots backtested — insufficient data overlap.")

In [ ]:
if len(bt_df) >= 5:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # 1. Markov P(YES) vs Market mid scatter
    ax = axes[0]
    colors = ["green" if o == 1 else "red" for o in bt_df["outcome"]]
    ax.scatter(bt_df["market_mid"], bt_df["p_markov"], c=colors, alpha=0.6, s=30, edgecolors="black", linewidth=0.3)
    ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("Market mid")
    ax.set_ylabel("Markov P(YES)")
    ax.set_title("Markov vs Market (green = UP won)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    # 2. Calibration: binned P(YES) vs actual frequency
    ax = axes[1]
    n_bins = min(10, len(bt_df) // 5)
    if n_bins >= 2:
        bt_df["p_bin"] = pd.cut(bt_df["p_markov"], bins=n_bins)
        cal = bt_df.groupby("p_bin", observed=True).agg(
            pred_mean=("p_markov", "mean"),
            actual_mean=("outcome", "mean"),
            count=("outcome", "count"),
        ).dropna()
        ax.plot(cal["pred_mean"], cal["actual_mean"], "o-", color="steelblue")
        ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, alpha=0.5)
        ax.set_xlabel("Mean predicted P(YES)")
        ax.set_ylabel("Actual UP rate")
        ax.set_title("Calibration curve")
    else:
        ax.text(0.5, 0.5, "Not enough data\nfor calibration", ha="center", va="center")

    # 3. P(YES) over time
    ax = axes[2]
    ax.plot(range(len(bt_df)), bt_df["p_markov"].values, label="Markov", color="steelblue")
    ax.plot(range(len(bt_df)), bt_df["market_mid"].values, label="Market", color="orange", alpha=0.7)
    for i, row in bt_df.iterrows():
        ax.scatter(i, row["outcome"], color="green" if row["outcome"] == 1 else "red",
                   s=15, zorder=5, alpha=0.5)
    ax.set_xlabel("Slot #")
    ax.set_ylabel("Probability")
    ax.set_title("P(YES) over slots (dots = actual outcome)")
    ax.legend()

    plt.tight_layout()
    plt.show()
else:
    print("Not enough slots for plots.")

In [ ]:
print("Done.")